# 03 — Train a DQN Model Online

Train a MOUSE model directly against live `mouse-env` environments. This is the preferred example when you want the agent to keep learning from fresh interaction instead of first writing a dataset and later replaying it offline.

The run alternates two phases each **cycle** until `GRADIENT_STEPS` optimizer updates have been applied (same total as `examples/02_train_offline.ipynb`):

1. **Rollout** — the current model (with epsilon-greedy exploration) interacts with live envs for up to `ENV_STEPS_PER_CYCLE` transitions. Each step is appended to an in-memory `Datastore`; per-env policy `contexts` feed the next action. Episode statistics for the segment just collected are printed at the end of each rollout.
2. **Train** — a `DataLoader` samples replay batches from those stores and the model is updated with `DqnObjective` for up to `GRADIENT_STEPS_PER_CYCLE` optimizer steps.

For held-out evaluation after training, use `examples/04_inference.ipynb`. Each phase is defined below; a master cell at the bottom runs rollout → train in a loop.


In [1]:
from collections import deque

import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead
from mouse_core.objectives import DqnObjective


MODEL_ID = "mouse-example-model"       # Hub repo id for push_model_to_hub
MAX_ACTIONS = 4                        # discrete action head size (FrozenLake: up/down/left/right)
MAX_OBS_DISCRETE = 64                  # observation embedder vocab (max cells in 8×8 maps)
NUM_ENVS = 1000                        # parallel SingleEnv instances (one Datastore each)

GRADIENT_STEPS = 20000                 # total optimizer updates (same budget as 02_train_offline)
SEQUENCE_LENGTH = 512                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                         # sequences per optimizer step

ENV_STEPS_PER_CYCLE = 1000             # env transitions collected each cycle
STEPS_PER_ENV = 100                    # consecutive env steps on one env before rotating
GRADIENT_STEPS_PER_CYCLE = 1000        # optimizer updates each cycle (once learning starts)

LEARNING_STARTS = 2000                 # env transitions before the first optimizer update
EXPLORATION_ENDS = 10000               # env step where epsilon ramp 0→1 finishes; stays at 1.0 after


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Online training keeps the environment in the training loop. Each `SingleEnv` instance runs one infinite task with deterministic FrozenLake dynamics and 50-step episode truncation, so the model keeps carrying policy context across episode boundaries.


In [2]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
    )
    for i in range(NUM_ENVS)
]

envs = [make_env(config) for config in configs]
print(f"Environments {min(10, len(envs))} of {len(envs)}:")
for env in envs[:10]:
    print(env.name)

Environments 10 of 1000:
proc_frozenlake_online_0
proc_frozenlake_online_1
proc_frozenlake_online_2
proc_frozenlake_online_3
proc_frozenlake_online_4
proc_frozenlake_online_5
proc_frozenlake_online_6
proc_frozenlake_online_7
proc_frozenlake_online_8
proc_frozenlake_online_9


## Build the model

The model uses a `StepEmbedder`, the full pretrained `Qwen3Backbone`, and a `DiscreteActionValueHead`.

For faster debugging you can pass `num_layers=...` to truncate the backbone.

In [3]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=10

## Replay buffer and policy contexts

Each env writes to one `Datastore`; together they are the live replay buffer. Each env also has a `contexts` deque capped at `SEQUENCE_LENGTH` — the action-selection history used during collection. Contexts are never cleared; episode boundaries appear as `done` values in the sequence.


In [4]:
stores = [Datastore(name=env.name) for env in envs]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in envs]


## Rollout phase

One **rollout** gathers up to `ENV_STEPS_PER_CYCLE` env transitions, visiting envs in order and taking up to `STEPS_PER_ENV` consecutive transitions per env before moving on. For each env visit the loop uses a single **B=1 KV cache** (like vLLM per-sequence caching): full-context pass on the first model call, then incremental passes for new rows only. The cache is discarded when moving to the next env — inference is never batched across envs, so only one cache is needed at a time.

1. Choose actions with epsilon-greedy exploration (`epsilon` is recomputed from the global env-step counter on every transition). Random actions skip the model; the next model call feeds all uncached rows as a catch-up slice.
2. Step the env, append to its `Datastore` and `contexts` deque. The KV cache grows incrementally for the duration of that env's visit and is discarded when the loop moves to the next env.

Each rollout clears every env tracker first; `print_rollout_stats` then reads completed episodes from the trackers when the rollout finishes.


In [5]:
def epsilon_for_env_step(env_step: int) -> float:
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    return min(env_step / EXPLORATION_ENDS, 1.0)


def print_rollout_stats(envs: list, env_steps: int, grad_steps: int, epsilon: float) -> None:
    """Print mean reward/length for episodes completed during the last rollout."""
    rewards: list[float] = []
    lengths: list[int] = []
    for env in envs:
        rewards.extend(env.tracker.episode_cum_rewards)
        lengths.extend(env.tracker.episode_lengths)

    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).float().mean().item() if n else float("nan")
    print(
        f"env_step={env_steps} grad_step={grad_steps} epsilon={epsilon:.3f} "
        f"| {n} completed episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}"
    )


In [6]:
def rollout(
    model: Model,
    envs: list,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
) -> int:
    """Collect up to ``ENV_STEPS_PER_CYCLE`` env transitions, then return."""
    for env in envs:
        env.tracker.clear()
    model.eval()

    budget = ENV_STEPS_PER_CYCLE
    collected = 0

    for env, store, context in zip(envs, stores, contexts):
        if collected >= budget:
            break

        kv_cache = None
        cache_count = 0
        ctx_count = 0
        chunk = min(STEPS_PER_ENV, budget - collected)

        for _ in range(chunk):
            epsilon = epsilon_for_env_step(env_steps)
            if not context or torch.rand(1).item() < epsilon:
                inp = env.sample_random_input()
            else:
                ctx_list = list(context)
                with torch.no_grad():
                    if kv_cache is None:
                        preds, _, kv_cache = model([ctx_list], use_cache=True)
                    else:
                        uncached = ctx_count - cache_count
                        preds, _, kv_cache = model(
                            [ctx_list[-uncached:]], cache=kv_cache, use_cache=True
                        )
                    cache_count = ctx_count

                action = model.get_action(preds, temperature=0.0, num_actions=MAX_ACTIONS)
                inp = {"action": action.squeeze().cpu()}

            out = env.step(inp)
            store.append({**inp, **out})
            context.append({**inp, **out})
            ctx_count += 1
            env_steps += 1
            collected += 1

    print_rollout_stats(envs, env_steps, grad_steps, epsilon_for_env_step(env_steps))
    return env_steps


## Training phase

One **train burst** runs after each rollout once `env_steps >= LEARNING_STARTS`:

1. Call `loader.refresh()` so the replay sampler re-snapshots stores and drops prefetched batches.
2. Run up to `GRADIENT_STEPS_PER_CYCLE` DQN updates (capped by remaining `GRADIENT_STEPS`).

The loader is created up front (stores may still be empty; with `pack=True`, sampling starts once rows have been appended). It uses `pack=True`, so stores shorter than `SEQUENCE_LENGTH` are padded by drawing additional segments. `weight_mode="per_step"` weights sampling toward larger stores. Pass `seed=...` for reproducible batches; default is non-deterministic.


In [7]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.99,
    gamma_episode_truncated=0.99,
    gamma_task_terminal=0.99,
    gamma_task_truncated=0.99,
    tau=0.0005,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

In [8]:
def train_burst(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: DqnObjective,
    loader: DataLoader,
    grad_steps: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``grad_steps`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(grad_steps):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)

    assert loss is not None
    return loss, metrics


## Run online training

The master loop alternates **rollout → train** until `GRADIENT_STEPS` optimizer updates have been applied — the same total as offline training. Optimizer updates are skipped until `LEARNING_STARTS` env transitions have been collected. Each rollout prints segment episode stats; training progress is logged every 1000 env transitions.


In [ ]:
env_steps = 0
grad_steps = 0
loss, metrics = None, {}

while grad_steps < GRADIENT_STEPS:
    env_steps = rollout(model, envs, stores, contexts, env_steps, grad_steps)

    if env_steps >= LEARNING_STARTS:
        burst = min(GRADIENT_STEPS_PER_CYCLE, GRADIENT_STEPS - grad_steps)
        loss, metrics = train_burst(model, optimizer, objective, loader, burst)
        grad_steps += burst

        if env_steps % 1000 == 0 and loss is not None:
            print(
                f"train loss={loss.item():.4f} q={metrics['q_values_mean']:.3f}"
            )

loader.close()

for env in envs:
    env.close()
print(f"Online training finished ({grad_steps} optimizer steps, {env_steps} env steps).")


env_step=1000 grad_step=0 epsilon=0.100 | 103 completed episodes | mean reward -0.053 | mean length 6.3
env_step=2000 grad_step=0 epsilon=0.200 | 109 completed episodes | mean reward -0.061 | mean length 8.9
train loss=0.0024 q=0.487
env_step=3000 grad_step=1000 epsilon=0.300 | 75 completed episodes | mean reward 0.148 | mean length 11.9


## Push to the Hub

Run this cell if you want to save the online-trained checkpoint to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
